# 060 — Site case-study MSA collapse fragilities

Collects the Multiple-Stripe-Analysis (MSA) collapse fragility of each case-study
structure from the analysis drive, copies it into the processed-data tree **with a
provenance manifest**, plots it against its stripe IMLs, and reports which curves are
*not well defined* — i.e. whose stripes never bracket collapse.

The processed copies written here are what notebook **061** consumes.

## Inputs

- `D:/07_wp1_casestudy_sites/site_{i}/{n}s/mdof/msa_AvgSA_03/collapse_fragility.json` —
  the MSA run output (`median`, `dispersion`, and the 2xN empirical curve `efc`).
- `data_processed/05_gcim_distributions/imls_for_selection_AvgSA_03.json` — the stripe
  IMLs, drawn on the plots as guide-lines.

## Outputs

| File | Content |
| --- | --- |
| `data_processed/09_structure_fragility_curves/wp1_casestudy_sites/site_{i}/{tag}_msa_collapsefragility_AvgSA_03.json` | the copied fragility (read by nb 061) |
| `.../{tag}_msa_collapsefragility_AvgSA_03.json.manifest.json` | its provenance sidecar |
| `results/05_site_fragility_curves/fragility_curves_site_{i}_{n}s.jpg` | the fragility plot |

## Caching

Each structure is cached against the **content hash of its source
`collapse_fragility.json` plus its stripe-IML list**, via
`cache_utils.json_load_or_compute`. A structure whose `.manifest.json` still matches is
left untouched — the JSON is not rewritten and the figure is not redrawn (unless the JPG
has gone missing). A re-run after re-running a handful of MSA analyses therefore
reprocesses only those. Set `FORCE_RECOMPUTE = True` to rebuild everything.

> **Run order.** The MSA results live on `D:` — run this on the machine with the drive
> attached to (re)build the processed copies. Without it the notebook prints a warning
> and falls back to the copies already in `data_processed/09_.../`, so the plots and the
> well-definedness report still work, but nothing is written.

In [3]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


## 0. Setup & parameters

In [4]:
import json
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import lognorm

from phd_project.config import config
from phd_project.scripts.cache_utils import fingerprint, json_load_or_compute

cfg = config.load_config()

In [5]:
# ----------------------------------------------------------------------------
# PARAMETERS
# ----------------------------------------------------------------------------
GM_SET = "AvgSA_03"
N_STOREYS = [3, 5]              # structure types to process
SITES = list(range(0, 60))      # case-study site indices

# --- FOLDERS ---
ANALYSIS_ROOT = cfg["analysis_data"]["wp1_casestudy_sites"]   # D: - the MSA run output
FRAG_ROOT = cfg["proc_data"]["wp1_sites_fragility_curves"]    # the processed JSON copies
RESULTS_ROOT = cfg["results"]["site_fragility_curves"]        # the fragility JPGs
STRIPE_IML_PATH = cfg["proc_data"][f"{GM_SET}_imls_for_selection"]

RESULTS_ROOT.mkdir(parents=True, exist_ok=True)

# --- WELL-DEFINEDNESS THRESHOLDS ---
# A curve is well defined only if its stripes bracket collapse: the empirical curve must
# rise ABOVE PC_UPPER_THRESHOLD and start BELOW PC_LOWER_THRESHOLD.
PC_UPPER_THRESHOLD = 0.7
PC_LOWER_THRESHOLD = 0.2

# Escape hatch: re-copy and re-plot every structure, ignoring the per-structure
# manifests. Leave False for normal incremental runs (only new/changed MSA runs rebuild).
FORCE_RECOMPUTE = False


def structure_tag(site_idx: int, n: int) -> str:
    return f"{n}s_cbf_dc2_site{site_idx}"


print(f"ANALYSIS_ROOT = {ANALYSIS_ROOT}")
print(f"FRAG_ROOT     = {FRAG_ROOT}")
print(f"RESULTS_ROOT  = {RESULTS_ROOT}")

ANALYSIS_ROOT = D:\07_wp1_casestudy_sites
FRAG_ROOT     = C:\Users\clemettn\Documents\phd\data_processed\09_structure_fragility_curves\wp1_casestudy_sites
RESULTS_ROOT  = C:\Users\clemettn\Documents\phd\results\05_site_fragility_curves


## 1. Stripe IMLs

The IMLs each structure's MSA stripes were run at, keyed by site index then structure
tag. They are drawn on the fragility plots as guide-lines and folded into the cache
fingerprint, so editing the IML file redraws the affected figures.

In [6]:
with open(STRIPE_IML_PATH, "r") as file:
    stripe_imls = json.load(file)

print(f"{len(stripe_imls)} sites in {STRIPE_IML_PATH.name}")

60 sites in imls_for_selection_AvgSA_03.json


## 2. Analysis-root availability

`ANALYSIS_ROOT` lives on the external `D:` drive. When it is not attached this is a
**warning, not an error**: the notebook switches to reading the already-processed copies
under `FRAG_ROOT` so the plots and the report below still run, and writes nothing.

In [7]:
ANALYSIS_OK = ANALYSIS_ROOT.exists()

if ANALYSIS_OK:
    print(f"Analysis root available: {ANALYSIS_ROOT}")
else:
    print(f"WARNING: analysis root {ANALYSIS_ROOT} is not accessible (drive not attached).")
    print(f"  Falling back to the processed copies in {FRAG_ROOT}.")
    print("  No fragility JSONs will be (re)written.")

Analysis root available: D:\07_wp1_casestudy_sites


## 3. Process the MSA fragility curves

For every `(site, storeys)`: resolve the source fragility on the analysis drive, pass it
through the provenance cache, run the well-definedness check, and redraw the figure only
when the cache was rebuilt (or the JPG is missing).

Statuses reported at the end:

- **computed** — source changed (or is new); JSON + manifest + JPG (re)written.
- **cached** — manifest matched; nothing rewritten.
- **offline** — read from the processed copy because `D:` is detached; nothing written.
- **skipped** — no MSA fragility available for that structure at all.

In [8]:
def fmt_sites(sites: list[int]) -> str:
    """Compact, sorted site listing for the printed summaries."""
    return ", ".join(str(s) for s in sorted(sites)) if sites else "-"


def msa_source_path(site: int, ns: int) -> Path:
    """The collapse fragility written by the MSA run, on the analysis drive."""
    return (ANALYSIS_ROOT / f"site_{site}" / f"{ns}s" / "mdof"
            / f"msa_{GM_SET}" / "collapse_fragility.json")


def processed_path(site: int, ns: int) -> Path:
    """The processed copy consumed by notebook 061."""
    tag = structure_tag(site, ns)
    return FRAG_ROOT / f"site_{site}" / f"{tag}_msa_collapsefragility_{GM_SET}.json"


def plot_path(site: int, ns: int) -> Path:
    return RESULTS_ROOT / f"fragility_curves_site_{site}_{ns}s.jpg"


def plot_fragility_curve(fc, building_stripe_imls, site, ns, out_fp):
    """Fitted lognormal + empirical MSA points, with the stripe IMLs marked."""
    im_max = lognorm.ppf(0.99, s=fc["dispersion"], scale=fc["median"])
    imls = np.linspace(0, im_max * 1.15, 50)   # in g
    msa_fc_fit = lognorm.cdf(imls, s=fc["dispersion"], scale=fc["median"])

    fig, ax = plt.subplots(figsize=(6, 4))
    plt.close(fig)

    for siml in building_stripe_imls:
        if siml is not None:
            ax.axvline(siml, ls="--", color="k", alpha=0.5)

    ax.plot(imls, msa_fc_fit, color="b", label="MSA")
    ax.plot(fc["efc"][0, :], fc["efc"][1, :], ls="none", marker="o", mfc="b", mec="k",
            alpha=0.75, label="MSA ecdf")

    ax.set_ylim(0, 1)
    ax.grid(ls="-.", color="0.8")
    ax.set_xlabel("AvgSA[0,3], [g]")
    ax.set_ylabel("Probability of Collapse, P[C]")
    ax.set_title(f"Fragility Curves - Site {site}, {ns}s")
    leg = ax.legend()
    leg.get_frame().set_edgecolor("k")

    fig.savefig(out_fp, dpi=300, bbox_inches="tight")

In [9]:
not_well_defined = {ns: {"lt_upper_threshold": [], "gt_lower_threshold": []}
                    for ns in N_STOREYS}
assessed = {ns: [] for ns in N_STOREYS}
statuses = {"computed": [], "cached": [], "offline": [], "skipped": []}

for site in SITES:
    for ns in N_STOREYS:
        tag = structure_tag(site, ns)
        save_fp = processed_path(site, ns)
        jpg_fp = plot_path(site, ns)
        building_stripe_imls = stripe_imls[str(site)][tag]

        if ANALYSIS_OK:
            src = msa_source_path(site, ns)
            if not src.is_file():
                statuses["skipped"].append((site, ns))
                continue

            # The cache key is the source fragility + this structure's stripe IMLs, so a
            # re-run MSA (or an edited IML list) is what invalidates the processed copy.
            save_fp.parent.mkdir(parents=True, exist_ok=True)
            fc_raw, status = json_load_or_compute(
                save_fp,
                fingerprint(msa_fragility=src, stripe_imls=building_stripe_imls),
                lambda src=src: json.load(open(src)),
                force=FORCE_RECOMPUTE,
                input_paths={"msa_fragility": src},
            )
        else:
            # Offline: reuse the processed copy as-is. Nothing is written and no manifest
            # is touched, so provenance stays whatever the last online run stamped.
            if not save_fp.is_file():
                statuses["skipped"].append((site, ns))
                continue
            with open(save_fp, "r") as file:
                fc_raw = json.load(file)
            status = "offline"

        statuses[status].append((site, ns))
        fc = {k: np.array(v) if isinstance(v, list) else v for k, v in fc_raw.items()}

        # --- well-definedness check: always runs, cached or not, so the report below is
        # complete on every run rather than only covering the rebuilt structures.
        assessed[ns].append(site)
        if max(fc["efc"][1, :]) <= PC_UPPER_THRESHOLD:
            not_well_defined[ns]["lt_upper_threshold"].append(site)
        if min(fc["efc"][1, :]) >= PC_LOWER_THRESHOLD:
            not_well_defined[ns]["gt_lower_threshold"].append(site)

        # --- plot only when the cache was rebuilt, or the figure has gone missing
        if status == "computed" or not jpg_fp.is_file():
            plot_fragility_curve(fc, building_stripe_imls, site, ns, jpg_fp)

print("\n" + "   ".join(f"[{name}] {len(v)}" for name, v in statuses.items()))
if statuses["skipped"]:
    print("  skipped (no MSA fragility available):")
    for ns in N_STOREYS:
        sk = [s for s, n in statuses["skipped"] if n == ns]
        if sk:
            listing = "all sites" if len(sk) == len(SITES) else fmt_sites(sk)
            print(f"    {ns}s - {len(sk)}: {listing}")


[computed] 114   [cached] 0   [offline] 0   [skipped] 6
  skipped (no MSA fragility available):
    3s - 3: 24, 31, 38
    5s - 3: 24, 31, 38


## 4. Fragility-curve definition check

Which structures the MSA stripes failed to bracket collapse for, **reported separately
for each storey count** — the two structure types have different periods and base-shear
coefficients, so their stripe placement fails in different ways and the counts are only
meaningful when kept apart.

In [10]:
print("Fragility-curve definition check")
print(f"  thresholds: max P[C] must exceed {PC_UPPER_THRESHOLD:.2f}, "
      f"min P[C] must fall below {PC_LOWER_THRESHOLD:.2f}")

for ns in N_STOREYS:
    n_assessed = len(assessed[ns])
    print(f"\n{ns}s structures - {n_assessed} assessed")

    if n_assessed == 0:
        print("  no MSA fragility curves found.")
        continue

    lt = not_well_defined[ns]["lt_upper_threshold"]
    gt = not_well_defined[ns]["gt_lower_threshold"]

    print(f"  max P[C] <= {PC_UPPER_THRESHOLD:.2f} (upper tail not reached) - {len(lt)} sites:")
    print(f"    {fmt_sites(lt)}")
    print(f"  min P[C] >= {PC_LOWER_THRESHOLD:.2f} (lower tail not reached) - {len(gt)} sites:")
    print(f"    {fmt_sites(gt)}")

    n_bad = len(set(lt) | set(gt))
    print(f"  well defined: {n_assessed - n_bad}/{n_assessed}  "
          f"({n_bad} site(s) fail at least one threshold)")

Fragility-curve definition check
  thresholds: max P[C] must exceed 0.70, min P[C] must fall below 0.20

3s structures - 57 assessed
  max P[C] <= 0.70 (upper tail not reached) - 20 sites:
    1, 2, 3, 4, 10, 13, 15, 16, 27, 30, 32, 33, 34, 35, 36, 42, 43, 48, 50, 53
  min P[C] >= 0.20 (lower tail not reached) - 12 sites:
    4, 12, 17, 22, 23, 27, 28, 37, 44, 49, 57, 58
  well defined: 27/57  (30 site(s) fail at least one threshold)

5s structures - 57 assessed
  max P[C] <= 0.70 (upper tail not reached) - 29 sites:
    0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 19, 20, 25, 26, 27, 29, 30, 34, 37, 44, 45, 46
  min P[C] >= 0.20 (lower tail not reached) - 16 sites:
    4, 12, 17, 18, 22, 23, 27, 28, 32, 33, 35, 36, 55, 57, 58, 59
  well defined: 15/57  (42 site(s) fail at least one threshold)
